# Module 22: Quantum PUF via IBM Quantum (Qiskit) — Lab

Theory: `theory.md` (QPUF design, Von Neumann entropy, and the quantum-gravity entropy chain).

**Objectives:**
1. Generate a quantum-random bitstream: a real IBM Quantum call if credentials/network are available, falling back gracefully to a local Qiskit Aer simulator, and finally to a numpy model of the same physics if `qiskit` itself isn't installed.
2. Run NIST-style statistical tests (frequency, runs) on that bitstream.
3. Compare it against Python's classical `random` module.
4. Sketch a QPUF fingerprint from per-qubit measurement statistics.

In [ ]:
import numpy as np
import random as py_random

np.random.seed(42)
py_random.seed(42)

N_SHOTS = 4096

## 1. Quantum-Random Bit Generation (Real Hardware → Aer Simulator → Numpy Fallback)

The circuit is the standard QRNG primitive: `H` (Hadamard) on `|0>` creates an equal
superposition `(|0> + |1>) / sqrt(2)`; measurement collapses it to 0 or 1 with 50/50
probability. Per `theory.md`, this collapse is believed to be *fundamentally*
non-deterministic (Von Neumann entropy of the post-measurement mixed state > 0), unlike
classical thermal noise (Module 04 §3.8), which is only *practically* unpredictable.

This cell tries three tiers, in order, and reports which one actually ran:
1. **Real IBM Quantum hardware** via `qiskit-ibm-runtime` (needs an API token + network — neither
   is available in most local/offline environments, including the one this lab was authored in).
2. **Local Qiskit Aer simulator** (needs `qiskit` + `qiskit-aer` installed, but no network/token).
3. **Numpy model of the same physics** (`Bernoulli(0.5)` per shot) — mathematically identical
   to an ideal Hadamard-and-measure circuit's output distribution, used only so this notebook
   runs end-to-end without any quantum-computing dependency installed.

In [ ]:
def qrng_ibm_hardware(n_shots):
    """Tier 1: run H+measure on real IBM Quantum hardware via qiskit-ibm-runtime.
    Requires QISKIT_IBM_TOKEN in the environment and network access."""
    import os
    from qiskit import QuantumCircuit
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

    token = os.environ.get("QISKIT_IBM_TOKEN")
    if not token:
        raise RuntimeError("No QISKIT_IBM_TOKEN set — skipping real hardware.")

    service = QiskitRuntimeService(channel="ibm_quantum", token=token)
    backend = service.least_busy(operational=True, simulator=False)

    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)

    sampler = Sampler(backend)
    job = sampler.run([qc], shots=n_shots)
    counts = job.result()[0].data.c.get_counts()
    bits = np.array([int(b) for b, c in counts.items() for _ in range(c)])
    np.random.shuffle(bits)
    return bits, f"IBM Quantum hardware ({backend.name})"


def qrng_aer_simulator(n_shots):
    """Tier 2: run the same circuit on a local Qiskit Aer simulator (no network/token needed)."""
    from qiskit import QuantumCircuit, transpile
    from qiskit_aer import AerSimulator

    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)

    sim = AerSimulator()
    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=n_shots).result()
    counts = result.get_counts()
    bits = np.array([int(b) for b, c in counts.items() for _ in range(c)])
    np.random.shuffle(bits)
    return bits, "Qiskit Aer (local quantum-circuit simulator)"


def qrng_numpy_fallback(n_shots):
    """Tier 3: pure-numpy model of an ideal Hadamard+measure circuit's output
    distribution (Bernoulli(0.5) per shot) — used when qiskit isn't installed at all."""
    bits = np.random.randint(0, 2, n_shots)
    return bits, "numpy Bernoulli(0.5) model (qiskit not installed — physics-equivalent fallback)"


def get_qrng_bits(n_shots):
    for fn in (qrng_ibm_hardware, qrng_aer_simulator, qrng_numpy_fallback):
        try:
            return fn(n_shots)
        except Exception as e:
            print(f"  [{fn.__name__}] unavailable: {e}")
    raise RuntimeError("All QRNG tiers failed — this should be unreachable (tier 3 has no external deps).")


print("Requesting quantum-random bits (tries real hardware, then simulator, then numpy fallback)...")
qrng_bits, source_used = get_qrng_bits(N_SHOTS)
print(f"\nSource actually used: {source_used}")
print(f"Collected {len(qrng_bits)} bits; fraction of 1s = {qrng_bits.mean():.4f} (ideal: 0.5000)")

## 2. NIST-Style Statistical Tests vs. Classical `random`

Same tests as Module 04's TRNG lab (`theory.md` §3.4/§3.7): frequency (monobit) and runs
tests, applied here to the quantum-sourced bitstream and to Python's classical
Mersenne-Twister `random` module for comparison.

In [ ]:
def frequency_test(bits):
    """Monobit frequency test (NIST SP 800-22 Test 1)."""
    n = len(bits)
    s = 2 * bits.sum() - n
    s_obs = abs(s) / np.sqrt(n)
    try:
        from scipy.special import erfc
    except ImportError:
        from math import erfc
        erfc = np.vectorize(erfc)
    p_value = erfc(s_obs / np.sqrt(2))
    return float(p_value)


def runs_test(bits):
    """Runs test (NIST SP 800-22 Test 2)."""
    n = len(bits)
    pi = bits.mean()
    if abs(pi - 0.5) >= 2 / np.sqrt(n):
        return 0.0
    runs = 1
    for i in range(1, n):
        if bits[i] != bits[i - 1]:
            runs += 1
    try:
        from scipy.special import erfc
    except ImportError:
        from math import erfc
        erfc = np.vectorize(erfc)
    num = abs(runs - 2 * n * pi * (1 - pi))
    den = 2 * np.sqrt(2 * n) * pi * (1 - pi)
    p_value = erfc(num / den)
    return float(p_value)


def min_entropy_bits(bits):
    """NIST SP 800-90B-style min-entropy (see Module 04 theory.md §3.8)."""
    p1 = bits.mean()
    p_max = max(p1, 1 - p1)
    return -np.log2(p_max)


classical_bits = np.array([py_random.getrandbits(1) for _ in range(N_SHOTS)])

print(f"{'Source':45s} {'Freq p-value':>14s} {'Runs p-value':>14s} {'Min-entropy':>12s}")
for name, bits in [(source_used, qrng_bits), ("Python random (Mersenne Twister, classical)", classical_bits)]:
    fp = frequency_test(bits)
    rp = runs_test(bits)
    hinf = min_entropy_bits(bits)
    print(f"{name:45s} {fp:14.4f} {rp:14.4f} {hinf:12.4f}")

print("\nInterpretation: p-values > 0.01 pass the corresponding NIST SP 800-22 test at the")
print("standard significance level. Both sources should pass here since both are designed")
print("to be close to fair-coin behavior; the qualitative difference between them (per")
print("theory.md) is *why* each is unpredictable — physical/quantum measurement collapse")
print("vs. a deterministic PRNG algorithm — not how they score on these particular tests.")

## 3. Sketch: QPUF Fingerprint from Per-Qubit Measurement Statistics

A real QPUF fingerprint (per `theory.md`, "Measurement-based QPUF") comes from *device-specific*
T1/T2 and gate-error variability across multiple qubits, not from the ideal 50/50 Hadamard
outcome alone. This cell simulates that variability across a small set of "qubits" with slightly
different (chip-specific) bias, and derives a hash fingerprint from the resulting counts —
mirroring the real workflow without requiring hardware access.

In [ ]:
import hashlib

N_QUBITS = 5
SHOTS_PER_QUBIT = 512

# Each simulated qubit has a slightly different bias due to device-specific gate error /
# readout noise (theory.md: "QPUF Entropy Sources") — an ideal qubit would be exactly 0.5.
device_biases = 0.5 + np.random.normal(0, 0.02, N_QUBITS)

fingerprint_bits = []
for qubit_id, bias in enumerate(device_biases):
    shots = (np.random.random(SHOTS_PER_QUBIT) < bias).astype(int)
    ones_fraction = shots.mean()
    print(f"qubit {qubit_id}: bias={bias:.4f}  measured P(1)={ones_fraction:.4f}")
    # 1 bit of the fingerprint per qubit: is this qubit biased above or below 0.5?
    fingerprint_bits.append(1 if ones_fraction > 0.5 else 0)

fingerprint_str = "".join(str(b) for b in fingerprint_bits)
fingerprint_hash = hashlib.sha256(fingerprint_str.encode()).hexdigest()[:16]
print(f"\nDevice fingerprint bits: {fingerprint_str}")
print(f"QPUF fingerprint hash:   {fingerprint_hash}")
print("\nRe-running this cell on the *same* simulated device (same device_biases) should")
print("reproduce a similar fingerprint; a different device (re-drawn device_biases) should not —")
print("this is the classical PUF reliability/uniqueness trade-off (Module 04) applied to a QPUF.")